### Basic two llm conversation using Openai and Ollama

In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()
#openai_client = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
requests.get("http://localhost:11434/").content

!ollama pull llama3.2

In [ ]:
!ollama list

In [ ]:
# Let's make a conversation between GPT-4.1-mini and ollama
# We're using cheap versions of models so the costs will be minimal

gpt_model = "gpt-4.1-mini"
#gemnini_model = "gemini-2.5-flash-lite"
ollma_model = "llama3.2:latest"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

ollma_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
ollma_messages = ["Hi"]

In [ ]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, ollma in zip(gpt_messages, ollma_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": ollma})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
def call_ollma():
    messages = [{"role": "system", "content": ollma_system}]
    for gpt, ollma_message in zip(gpt_messages, ollma_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": ollma_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = ollama.chat.completions.create(model=ollma_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
gpt_messages = ["Hi there"]
ollma_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Ollma:\n{ollma_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    ollma_next = call_ollma()
    display(Markdown(f"### Ollma:\n{ollma_next}\n"))
    ollma_messages.append(ollma_next)